In [1]:
"""
DATA_SAMPLER.IPYNB - STANDALONE DATA SAMPLING SCRIPT
=====================================================
Place this in a separate notebook (data_sampler.ipynb)
Run it once to create sampled versions of your datasets.

This will:
1. Read original CSV files from 'data/' directory
2. Create proportional samples maintaining train/test ratios
3. Save sampled files back to 'data/' directory (overwrites originals)
4. Create backup of original file sizes in 'backup_info.txt'

IMPORTANT: Make sure you have backups of your original data before running!
"""

import pandas as pd
import numpy as np
import os
from datetime import datetime

print("="*70)
print("SOLAR FLARE DATA SAMPLER - STANDALONE VERSION")
print("="*70)
print(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# =============================================================================
# CONFIGURATION - ADJUST THESE VALUES
# =============================================================================

# Set your desired TRAINING sample size
TRAIN_SAMPLE_SIZE = 40000  # Recommended: 10000-20000 for testing
                           # Set to None to use full dataset

# Maintain class balance?
BALANCED = True  # True = 50/50 flare/no-flare, False = preserve original distribution

# Directories
INPUT_DIR = 'data'          # Where original CSV files are located
OUTPUT_DIR = 'data'         # Where to save sampled files (same as input = overwrite)
BACKUP_INFO = True          # Create backup info file with original sizes

print(f"\n{'='*70}")
print("CONFIGURATION")
print(f"{'='*70}")
print(f"Training sample size: {TRAIN_SAMPLE_SIZE:,} rows" if TRAIN_SAMPLE_SIZE else "Using FULL dataset (no sampling)")
print(f"Test sample size: Proportional to training")
print(f"Balanced sampling: {BALANCED}")
print(f"Input directory: {INPUT_DIR}/")
print(f"Output directory: {OUTPUT_DIR}/")

if INPUT_DIR == OUTPUT_DIR and TRAIN_SAMPLE_SIZE is not None:
    print(f"\n⚠️  WARNING: This will OVERWRITE original files in '{OUTPUT_DIR}/'")
    print("   Make sure you have backups!")

# =============================================================================
# CREATE DIRECTORIES
# =============================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n✓ Output directory ready: {OUTPUT_DIR}/")

# =============================================================================
# FILE GROUPS (TRAIN/TEST PAIRS)
# =============================================================================

FILE_GROUPS = [
    {
        'train': 'training_data_C_24.csv',
        'test': 'testing_data_C_24.csv',
        'class': 'C'
    },
    {
        'train': 'training_data_M_24.csv',
        'test': 'testing_data_M_24.csv',
        'class': 'M'
    },
    {
        'train': 'training_data_M5_24.csv',
        'test': 'testing_data_M5_24.csv',
        'class': 'M5'
    }
]

# =============================================================================
# SAMPLING FUNCTION
# =============================================================================

def sample_dataset(filename, sample_size, balanced=True):
    """
    Sample dataset while preserving or balancing class distribution
    
    Parameters:
    -----------
    filename : str
        Full path to CSV file
    sample_size : int or None
        Number of samples to keep (None = use all data)
    balanced : bool
        If True, maintain 50/50 class balance
    
    Returns:
    --------
    tuple: (sampled_df, original_size, n_positive, n_negative)
    """
    
    # Load data
    df = pd.read_csv(filename)
    original_size = len(df)
    
    # If no sampling requested, return original
    if sample_size is None:
        label_col = df.columns[0]
        df['is_flare'] = df[label_col].apply(lambda x: 0 if str(x)[0].upper() == 'N' else 1)
        n_positive = df['is_flare'].sum()
        n_negative = len(df) - n_positive
        df = df.drop('is_flare', axis=1)
        return df, original_size, n_positive, n_negative
    
    # Identify label column (first column)
    label_col = df.columns[0]
    
    # Create binary label
    df['is_flare'] = df[label_col].apply(lambda x: 0 if str(x)[0].upper() == 'N' else 1)
    
    n_positive = df['is_flare'].sum()
    n_negative = len(df) - n_positive
    
    # Sample based on strategy
    if balanced:
        # Equal samples from each class
        samples_per_class = sample_size // 2
        
        positive_samples = df[df['is_flare'] == 1].sample(
            n=min(samples_per_class, n_positive), 
            random_state=42
        )
        negative_samples = df[df['is_flare'] == 0].sample(
            n=min(samples_per_class, n_negative), 
            random_state=42
        )
        
        sampled_df = pd.concat([positive_samples, negative_samples])
        sampled_df = sampled_df.sample(frac=1, random_state=42)  # Shuffle
        
    else:
        # Preserve original class distribution
        sampled_df = df.sample(n=min(sample_size, original_size), random_state=42)
    
    # Remove helper column
    sampled_df = sampled_df.drop('is_flare', axis=1)
    
    return sampled_df, original_size, n_positive, n_negative

# =============================================================================
# BACKUP ORIGINAL FILE INFO
# =============================================================================

if BACKUP_INFO and TRAIN_SAMPLE_SIZE is not None:
    backup_file = 'original_data_info.txt'
    with open(backup_file, 'w') as f:
        f.write("="*70 + "\n")
        f.write("ORIGINAL DATA FILE INFORMATION\n")
        f.write("="*70 + "\n")
        f.write(f"Backup created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        for group in FILE_GROUPS:
            train_path = os.path.join(INPUT_DIR, group['train'])
            test_path = os.path.join(INPUT_DIR, group['test'])
            
            if os.path.exists(train_path) and os.path.exists(test_path):
                train_size = len(pd.read_csv(train_path))
                test_size = len(pd.read_csv(test_path))
                
                f.write(f"{group['class']}-CLASS:\n")
                f.write(f"  {group['train']}: {train_size:,} rows\n")
                f.write(f"  {group['test']}: {test_size:,} rows\n\n")
    
    print(f"✓ Created backup info file: {backup_file}")

# =============================================================================
# PROCESS ALL FILE GROUPS
# =============================================================================

print(f"\n{'='*70}")
print("PROCESSING DATASETS")
print(f"{'='*70}")

results = []

for group in FILE_GROUPS:
    flare_class = group['class']
    train_file = group['train']
    test_file = group['test']
    
    train_path = os.path.join(INPUT_DIR, train_file)
    test_path = os.path.join(INPUT_DIR, test_file)
    
    print(f"\n{'='*70}")
    print(f"PROCESSING {flare_class}-CLASS DATA")
    print(f"{'='*70}")
    
    # Check if files exist
    if not os.path.exists(train_path) or not os.path.exists(test_path):
        print(f"⚠️  WARNING: Files not found, skipping {flare_class}-class")
        print(f"   Looking for: {train_path}")
        continue
    
    # Sample training data
    print(f"\nTraining Data: {train_file}")
    print(f"{'-'*70}")
    train_sampled, train_orig_size, train_pos, train_neg = sample_dataset(
        train_path, TRAIN_SAMPLE_SIZE, BALANCED
    )
    print(f"Original: {train_orig_size:,} rows (Pos: {train_pos:,}, Neg: {train_neg:,})")
    
    if TRAIN_SAMPLE_SIZE:
        print(f"Sampled:  {len(train_sampled):,} rows ({100*len(train_sampled)/train_orig_size:.1f}% of original)")
    else:
        print(f"Using full dataset (no sampling)")
    
    # Calculate proportional test size
    test_df_temp = pd.read_csv(test_path)
    original_ratio = len(test_df_temp) / train_orig_size
    
    if TRAIN_SAMPLE_SIZE:
        test_sample_size = int(TRAIN_SAMPLE_SIZE * original_ratio)
    else:
        test_sample_size = None
    
    # Sample testing data
    print(f"\nTesting Data: {test_file}")
    print(f"{'-'*70}")
    test_sampled, test_orig_size, test_pos, test_neg = sample_dataset(
        test_path, test_sample_size, BALANCED
    )
    print(f"Original: {test_orig_size:,} rows (Pos: {test_pos:,}, Neg: {test_neg:,})")
    
    if test_sample_size:
        print(f"Sampled:  {len(test_sampled):,} rows ({100*len(test_sampled)/test_orig_size:.1f}% of original)")
        print(f"Train:Test ratio: {len(train_sampled)/len(test_sampled):.2f}:1")
    else:
        print(f"Using full dataset (no sampling)")
        print(f"Train:Test ratio: {len(train_sampled)/len(test_sampled):.2f}:1")
    
    # Save data
    train_output = os.path.join(OUTPUT_DIR, train_file)
    test_output = os.path.join(OUTPUT_DIR, test_file)
    
    train_sampled.to_csv(train_output, index=False)
    test_sampled.to_csv(test_output, index=False)
    
    print(f"\n✓ Saved training data: {train_output}")
    print(f"✓ Saved testing data:  {test_output}")
    
    results.append({
        'class': flare_class,
        'train_orig': train_orig_size,
        'train_sampled': len(train_sampled),
        'test_orig': test_orig_size,
        'test_sampled': len(test_sampled)
    })

# =============================================================================
# SUMMARY REPORT
# =============================================================================

print(f"\n{'='*70}")
print("PROCESSING COMPLETE - SUMMARY")
print(f"{'='*70}\n")

if results:
    print(f"{'Class':<8} {'Train Original':<15} {'Train Sampled':<15} {'Test Original':<15} {'Test Sampled':<15}")
    print(f"{'-'*75}")
    
    for r in results:
        print(f"{r['class']:<8} {r['train_orig']:<15,} {r['train_sampled']:<15,} {r['test_orig']:<15,} {r['test_sampled']:<15,}")
    
    total_train_orig = sum(r['train_orig'] for r in results)
    total_train_samp = sum(r['train_sampled'] for r in results)
    total_test_orig = sum(r['test_orig'] for r in results)
    total_test_samp = sum(r['test_sampled'] for r in results)
    
    print(f"{'-'*75}")
    print(f"{'TOTAL':<8} {total_train_orig:<15,} {total_train_samp:<15,} {total_test_orig:<15,} {total_test_samp:<15,}")
    
    if TRAIN_SAMPLE_SIZE:
        reduction = 100 * (1 - (total_train_samp + total_test_samp) / (total_train_orig + total_test_orig))
        print(f"\nOverall data reduction: {reduction:.1f}%")
        
        print(f"\n{'='*70}")
        print("ESTIMATED TRAINING TIMES (with sampled data):")
        print(f"{'='*70}")
        print(f"  My ML Code (RF/KNN/NN):     ~5-10 minutes total")
        print(f"  SolarFlareNet (per class):  ~10-20 minutes")
        print(f"  SolarFlareNet (all 3):      ~30-60 minutes total")
    else:
        print(f"\nNo sampling applied - using full datasets")
    
    print(f"\n{'='*70}")
    print("NEXT STEPS:")
    print(f"{'='*70}")
    print("1. Your data files in 'data/' directory are now ready")
    print("2. Run your ML notebooks normally - they'll automatically use the sampled data")
    print("3. To restore original data, re-upload from your backups")
    
    if BACKUP_INFO and TRAIN_SAMPLE_SIZE:
        print(f"4. Original file sizes saved in: {backup_file}")
    
else:
    print("⚠️  No files were processed.")
    print(f"   Make sure CSV files exist in '{INPUT_DIR}/' directory")

print(f"\n{'='*70}")
print("DATA SAMPLING COMPLETE!")
print(f"{'='*70}")

SOLAR FLARE DATA SAMPLER - STANDALONE VERSION
Run timestamp: 2025-12-13 23:49:48

CONFIGURATION
Training sample size: 40,000 rows
Test sample size: Proportional to training
Balanced sampling: True
Input directory: data/
Output directory: data/

⚠️  WARNING: This will OVERWRITE original files in 'data/'
   Make sure you have backups!

✓ Output directory ready: data/
✓ Created backup info file: original_data_info.txt

PROCESSING DATASETS

PROCESSING C-CLASS DATA

Training Data: training_data_C_24.csv
----------------------------------------------------------------------
Original: 179,647 rows (Pos: 153,122, Neg: 26,525)
Sampled:  40,000 rows (22.3% of original)

Testing Data: testing_data_C_24.csv
----------------------------------------------------------------------
Original: 62,560 rows (Pos: 46,682, Neg: 15,878)
Sampled:  13,928 rows (22.3% of original)
Train:Test ratio: 2.87:1

✓ Saved training data: data/training_data_C_24.csv
✓ Saved testing data:  data/testing_data_C_24.csv

PROCE